In [1]:
import requests as rq
from botocore.exceptions import ClientError
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv
import os

from tornado.web import create_signed_value

# Load environment variables from .env file
load_dotenv()

True

In [4]:
''' ************FIRMS DATA************ '''

# User credentials
username = os.getenv("user")
password = os.getenv("passw")
MAP_KEY = os.getenv("map_key")
# MODIS 7 -day data for USA
url = f'https://firms.modaps.eosdis.nasa.gov/api/country/csv/{MAP_KEY}/VIIRS_NOAA21_NRT/USA/7'

response = rq.get(url, auth=HTTPBasicAuth(username, password))

# save the data
if response.status_code == 200:
    with open("VIIRS_NOAA21_NRT_usa_7day.csv", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("✅ Data saved to VIIRS_NOAA21_NRT_usa_7day.csv")

else:
    print(f"❌ Failed to fetch data: {response.status_code} - {response.text}")

✅ Data saved to VIIRS_NOAA21_NRT_usa_7day.csv


In [5]:
import pandas as pd

df = pd.read_csv("VIIRS_NOAA21_NRT_usa_7day.csv")
col = df.columns.tolist()
print(col)

['country_id', 'latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight']


In [9]:
df.head()

,country_id,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,USA,63.52622,-152.13304,367.00,0.60,0.70,2025-07-03,9,N21,VIIRS,h,2.0NRT,294.99,18.82,D
1,USA,64.67680,-148.47678,344.96,0.76,0.77,2025-07-03,9,N21,VIIRS,n,2.0NRT,292.13,12.11,D
2,USA,64.67944,-148.44040,329.61,0.76,0.77,2025-07-03,9,N21,VIIRS,n,2.0NRT,291.42,14.45,D
3,USA,64.68022,-148.45395,339.53,0.76,0.77,2025-07-03,9,N21,VIIRS,n,2.0NRT,292.72,18.28,D
4,USA,64.68104,-148.46817,331.61,0.76,0.77,2025-07-03,9,N21,VIIRS,n,2.0NRT,292.34,18.28,D


In [8]:
''' ************OPEN_WHEATHER DATA************ '''

API_KEY = os.getenv("api_key")

lat = 37.7749 # example latitude (San Francisco)
lon = -122. # example longitude
url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"

response = rq.get(url)
print(response.json())

{'coord': {'lon': -122, 'lat': 37.7749}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 13.19, 'feels_like': 12.89, 'temp_min': 13.19, 'temp_max': 13.19, 'pressure': 1019, 'humidity': 89, 'sea_level': 1019, 'grnd_level': 993}, 'visibility': 10000, 'wind': {'speed': 1.38, 'deg': 226, 'gust': 1.27}, 'clouds': {'all': 0}, 'dt': 1752053811, 'sys': {'country': 'US', 'sunrise': 1752065649, 'sunset': 1752118319}, 'timezone': -25200, 'id': 5392593, 'name': 'San Ramon', 'cod': 200}


# **LAMDA 1**

In [4]:
import boto3
import requests as rq
from datetime import datetime
import os

MAP_KEY = os.getenv("map_key")
COUNTRY_CODE = "USA"
today = datetime.today().strftime("%Y-%m-%d")

# API URL
url = f"https://firms.modaps.eosdis.nasa.gov/api/country/csv/{MAP_KEY}/VIIRS_NOAA21_NRT/{COUNTRY_CODE}/7/{today}"

# Make request
response = rq.get(url, timeout=10)

if response.status_code == 200 and "Invalid" not in response.text:
    print("✅ Data fetched successfully.")

    # Make sure directory exists
    os.makedirs("data-raw", exist_ok=True)


    # Save to /tmp (Lambda-style)
    file_name = f"data-raw/viirs_wildfire_{today}.csv"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(response.text)

    # Upload to S3
    s3 = boto3.client("s3")
    s3.upload_file(
        Filename=file_name,
        Bucket="wildfire-risk-data-sfg",
        Key=f"raw/wildfires/{today}.csv"
    )
    print("✅ File uploaded to S3.")
else:
    print("❌ Failed to fetch data.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)


✅ Data fetched successfully.
✅ File uploaded to S3.


## **LAMDA 2**

In [9]:
import boto3 , os , csv , json
import requests as rq
from datetime import datetime
from io import StringIO

def lambda_handler(event, context):
    WEATHER_API_KEY = os.getenv("api_key")
    BUCKET_NAME = "wildfire-risk-data-sfg"
    today = datetime.today().strftime("%Y-%m-%d")
    wildfire_key = f"raw/wildfires/{today}.csv"

    s3 = boto3.client("s3")
    try:
        response = s3.get_object(Bucket=BUCKET_NAME, Key=wildfire_key)
        csv_content = response["Body"].read().decode("utf-8")
        print(csv_content)
        print("✅ Wildfire data fetched from S3.")
    except Exception as e:
        print("❌ Error fetching CSV from S3:", e)
        return

lambda_handler(None,None)


country_id,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
USA,65.2241,-153.38522,367,0.67,0.74,2025-07-13,22,N21,VIIRS,h,2.0NRT,295.66,74.51,D
USA,65.22434,-153.3718,335.95,0.67,0.74,2025-07-13,22,N21,VIIRS,n,2.0NRT,289.95,39.28,D
USA,65.2251,-153.38704,367,0.67,0.74,2025-07-13,22,N21,VIIRS,h,2.0NRT,298.57,56.33,D
USA,65.22514,-153.40788,332.12,0.67,0.74,2025-07-13,22,N21,VIIRS,n,2.0NRT,290.64,74.51,D
USA,65.22622,-153.40999,367,0.67,0.74,2025-07-13,22,N21,VIIRS,h,2.0NRT,291.85,56.33,D
USA,65.22701,-153.4263,352.63,0.67,0.74,2025-07-13,22,N21,VIIRS,n,2.0NRT,296.91,89.76,D
USA,65.23026,-153.37044,331.73,0.67,0.74,2025-07-13,22,N21,VIIRS,n,2.0NRT,288.26,46.2,D
USA,65.23078,-153.3817,367,0.67,0.74,2025-07-13,22,N21,VIIRS,h,2.0NRT,291.33,74.51,D
USA,65.23164,-153.4003,367,0.67,0.74,2025-07-13,22,N21,VIIRS,h,2.0NRT,298.58,74.51,D
USA,65.23179,-153.384,345.1,0.67,0.74,2025-07-13,22,N21,VIIRS,n,2.0NRT,290.03,121.28,D



#### **_csv_content = response["Body"]_**

<botocore.httpchecksum.StreamingChecksumBody object at 0x000001DC18B41480>